# NB01: London Air Quality Data Collection

**DS105W Mini-Project 1, Data for Data Science (Winter Term 2025/2026)**

<div style="font-family: system-ui; padding: 20px 30px 20px 20px; background-color: #FFFFFF; border-left: 8px solid #ED9255; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);max-width:600px;color:#212121;">

**Student Notebook**
- 📅 Date: 23 February 2026
- 👤 Name: Joshua Andrew
- 📛 Candidate Number: 67321
- 🎯 Purpose: Collect historical hourly air pollution data for London from the OpenWeather API and save it to a JSON file.
- ❓  **Question: Does London's air clean up on weekends?**

</div>

## Overview

This notebook collects historical air pollution data for central London using the [OpenWeather Air Pollution API](https://openweathermap.org/api/air-pollution). The data will be saved as a JSON file in the `data/` folder so that later notebooks (`NB02`, `NB03`) can work from the saved file without needing to call the API again.

This follows the same workflow from 🖥️ [W03 Lecture](https://lse-dsi.github.io/DS105/2025-2026/winter-term/weeks/week03/lecture.html) and the 📝 [W04 Practice](https://moodle.lse.ac.uk/mod/page/view.php?id=1597988) NB01. 1. Make an API request. 2. Check the response. 3. Save the result locally.

## Reproducibility

To run this notebook you need an OpenWeather API key stored in a `.env` file at the root of this repository:

```text
API_KEY=your_api_key_here
```

The `.env` file is `.gitignore`-d so it will never be committed. You can get a free API key by creating an account at [OpenWeather](https://home.openweathermap.org/users/sign_up). The `python-dotenv` package is also needed (`pip install python-dotenv`).

## Section 1: Setup

⚙️ **Importing libraries**

- `requests`: makes HTTP requests to web APIs (introduced in 🖥️ [W02 Lecture](https://lse-dsi.github.io/DS105/2025-2026/winter-term/weeks/week02/lecture.html))
- `json`: reads and writes JSON files (used in 🖥️ [W03 Lecture](https://lse-dsi.github.io/DS105/2025-2026/winter-term/weeks/week03/lecture.html))
- `os`: creates directories on disk
- `datetime`: converts dates to UNIX timestamps for the API
- `dotenv`: loads the API key from the `.env` file without exposing it in code

In [1]:
import os
import json
import requests
from datetime import datetime

from dotenv import load_dotenv

print("✅ Libraries loaded successfully!")

✅ Libraries loaded successfully!


## Section 2: API Authentication

The OpenWeather API requires an API key for every request, unlike the Open-Meteo API we used in the W03 Lab and W04 Practice. I am loading the key from a `.env` file using `python-dotenv` so that it stays out of the code and never gets pushed to GitHub.

In [2]:
load_dotenv()

# As long as I don't `print(api_key)`, no one will ever be able to see my key

api_key = os.getenv("API_KEY")

#This line is to check the first 5 characters of the API Key to ensure that the .env file is readable
print(f"The first 5 characters of the API key: {api_key[:5]}")


The first 5 characters of the API key: 0dfd3


💭 **Personal Reflection Notes:**

This is the first time I have used `python-dotenv` and `.env` files. In previous weeks (W03 Lab, W04 Practice) we used Open-Meteo which did not need any authentication at all. I followed the setup instructions in the NB01 template and the [python-dotenv documentation](https://pypi.org/project/python-dotenv/) to understand how `load_dotenv()` reads the `.env` file into environment variables. It took me a moment to realise the key had to be pasted with no extra spaces or quotes in the `.env` file, because my first attempt gave me an authentication error (status 401). Once I fixed that the key loaded fine.

Before collecting a large amount of data, I want to test that authentication works with a quick call to the current air pollution endpoint.

In [3]:
# Quick test: get current air pollution for London
url = "http://api.openweathermap.org/data/2.5/air_pollution"

test_params = {
    "lat": 51.5074,
    "lon": -0.1278,
    "appid": api_key
}

test_response = requests.get(url, params=test_params)

print(f"Status code: {test_response.status_code}")

Status code: 200


In [4]:
# Check what the response looks like
if test_response.status_code == 200:
    print("Authentication works!")
    current_data = test_response.json()
    print(f"Keys: {list(current_data.keys())}")
    print(f"Number of records: {len(current_data['list'])}")
    print(f"\nSample record:")
    print(current_data["list"][0])
else:
    print("Request failed!")
    print(test_response.json())

Authentication works!
Keys: ['coord', 'list']
Number of records: 1

Sample record:
{'main': {'aqi': 2}, 'components': {'co': 180.81, 'no': 0.02, 'no2': 26.26, 'o3': 34.09, 'so2': 6.93, 'pm2_5': 14.93, 'pm10': 23.06, 'nh3': 0.36}, 'dt': 1772077933}


💭 **Personal Reflection Notes:**

I ran this test call before attempting the full historical download to make sure everything was working. The response structure is nested. Each record in the `"list"` array has `"dt"` (UNIX timestamp), `"main"` (overall AQI, 1-5 scale), and `"components"` (individual pollutant concentrations). I will need to flatten this in NB02.

I also noticed that the current endpoint (`/air_pollution`) only returns one point in time. For historical data I need a different endpoint (`/air_pollution/history`) that takes `start` and `end` UNIX timestamps. I figured this out by reading through the different endpoints on the [API documentation page](https://openweathermap.org/api/air-pollution). It was not immediately obvious which endpoint to use since there are several listed.

## Section 3: Collect Historical Data

The [Air Pollution History API](https://openweathermap.org/api/air-pollution) returns hourly pollution data for a given location between a `start` and `end` time (both as UNIX timestamps). Historical data is available from 27 November 2020 onwards.

💭 **Personal Reflection Notes:**

**Decisions I made here:**

- **Time period:** I chose to collect all available data from the API, starting from 27 November 2020 (the earliest date the API supports) through to the end of 2025. This gives me roughly five years of hourly data, which should be plenty for comparing weekday and weekend patterns. Having multiple years also lets me check whether any patterns are consistent over time or just a one-off. One thing to keep in mind is that 2020 and 2021 overlap with COVID lockdowns, when traffic was much lower than normal. I will need to think about how to handle this in NB03. I could either filter those periods out or at least note them down when interpreting the results.
- **Location:** I am using the coordinates for central London (51.5074, -0.1278), which is roughly around Westminster. The project asks about "London's air" generally, so a central location felt like a reasonable choice. I could compare multiple locations if I wanted to observe differences between areas of London, but one central point is enough to answer the main question.
- **Single API call:** The history endpoint accepts a start and end timestamp and returns all the hourly data in between. This means I can get the full range in one `requests.get()` call, which is exactly the same pattern as the Open-Meteo request in the W04 Practice NB01 and the W03 Lab notebook. There is no need for a loop here.

In [5]:
# Convert my chosen date range to UNIX timestamps
# The API requires these as integers (seconds since 1 January 1970)
# Historical data is available from 27 November 2020 onwards

start_date = datetime(2020, 11, 27)
end_date = datetime(2025, 12, 31, 23, 59, 59)

start_unix = int(start_date.timestamp())
end_unix = int(end_date.timestamp())

print(f"Start: {start_date} -> {start_unix}")
print(f"End:   {end_date} -> {end_unix}")

Start: 2020-11-27 00:00:00 -> 1606435200
End:   2025-12-31 23:59:59 -> 1767225599


💭 **Personal Reflection Notes:**

I used `datetime` from the Python standard library to convert dates into UNIX timestamps. I had not used `datetime.timestamp()` before, but the project brief lists `datetime` as one of the packages to use, so I looked it up in the [Python docs](https://docs.python.org/3/library/datetime.html#datetime.datetime.timestamp). The API documentation examples show UNIX timestamps like `1606223802`, and I checked that my converted values looked reasonable by comparing them to an online UNIX timestamp converter.

In [6]:
# Collect historical air pollution data in a single request
base_url = "http://api.openweathermap.org/data/2.5/air_pollution/history"

params = {
    "lat": 51.5074,
    "lon": -0.1278,
    "start": start_unix,
    "end": end_unix,
    "appid": api_key
}

response = requests.get(base_url, params=params)

print(f"Status code: {response.status_code}")

Status code: 200


In [7]:
if response.status_code == 200:
    data = response.json()
    print("✅ Data collected successfully!")
    print(f"Keys in response: {list(data.keys())}")
    print(f"Number of hourly records: {len(data['list'])}")
else:
    print("The API didn't return any data!")
    print(response.json())

✅ Data collected successfully!
Keys in response: ['coord', 'list']
Number of hourly records: 44064


## Section 4: Inspect the Data

Before saving, I want to check the data to make sure it covers the right time range and has the structure I expect.

In [8]:
# Look at the first record
print("First record:")
print(data["list"][0])

First record:
{'main': {'aqi': 2}, 'components': {'co': 347.14, 'no': 33.53, 'no2': 41.13, 'o3': 0.01, 'so2': 7.51, 'pm2_5': 18.81, 'pm10': 21.35, 'nh3': 0.25}, 'dt': 1606435200}


In [9]:
# Look at the last record
print("Last record:")
print(data["list"][-1])

Last record:
{'main': {'aqi': 2}, 'components': {'co': 131.29, 'no': 0, 'no2': 6.93, 'o3': 79.17, 'so2': 3.14, 'pm2_5': 3.3, 'pm10': 4.98, 'nh3': 0.23}, 'dt': 1767222000}


In [10]:
# Convert first and last timestamps to readable dates to confirm range
first_dt = datetime.fromtimestamp(data["list"][0]["dt"])
last_dt = datetime.fromtimestamp(data["list"][-1]["dt"])

print(f"Data starts: {first_dt}")
print(f"Data ends:   {last_dt}")
print(f"Total hourly records: {len(data['list'])}")

Data starts: 2020-11-27 00:00:00
Data ends:   2025-12-31 23:00:00
Total hourly records: 44064


In [11]:
# Check what pollutants are available
print("Available pollutant fields:")
print(list(data["list"][0]["components"].keys()))

Available pollutant fields:
['co', 'no', 'no2', 'o3', 'so2', 'pm2_5', 'pm10', 'nh3']


💭 **Personal Reflection Notes:**

I checked the first and last records to confirm the data covers the full range from late November 2020 through to the end of 2025. This felt important to do before saving because if the date range had been wrong I would have had to redo the API call. The record count (44,064) is close to but slightly below the expected 44,664 hours (1,861 days × 24), which suggests there are some gaps in the data where the API has no readings. This is normal for real-world data and should not affect the overall weekday vs weekend comparison since the gaps are likely spread across both. I will try and use pandas in NB02 to identify where the data gaps are and if they would affect the analysis. The 8 available pollutants (CO, NO, NO2, O3, SO2, PM2.5, PM10, NH3) give me plenty of options for the weekday vs weekend comparison in NB03. I will decide which ones to focus on in NB02 once I can see the data in a DataFrame.

## Section 5: Save to JSON

Following the pattern from 🖥️ [W03 Lecture](https://lse-dsi.github.io/DS105/2025-2026/winter-term/weeks/week03/lecture.html) and the W04 Practice NB01 solution, I am saving the full API response as a JSON file. This way NB02 can work entirely from the saved file.

In [12]:
# Make sure the data folder exists
os.makedirs("../data", exist_ok=True)

# Save the full response
with open("../data/london_air_pollution_2020_2025.json", "w") as f:
    json.dump(data, f, indent=2)

print("✅ Saved to data/london_air_pollution_2020_2025.json")

✅ Saved to data/london_air_pollution_2020_2025.json


💭 **Personal Reflection Notes:**

I saved the entire API response rather than extracting just specific pollutant values. This way, if I later decide to look at different pollutants or the AQI index itself, I already have everything and will not need to call the API again. The file is a few megabytes at most, so the tradeoff between storage and flexibility seemed worth it. The `json.dump()` with `indent=2` and `os.makedirs("data", exist_ok=True)` pattern is the same one from the W04 Practice NB01 solution and the W03 Lecture file-saving examples. I used `indent=2` to keep the file readable in case I need to open it manually to debug anything in NB02.

NB01 is done. Data transformation and analysis will be done in NB02 and NB03.